In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from warnings import warn

In [ ]:
# ======================================================
# COTTON Yield Processing with Unit Normalization
# ======================================================

# -----------------------
# 1. Load raw cotton data
# -----------------------
cotton_df = pd.read_csv("Data/Raw_Cotton.csv")

# Clean Value column
cotton_df["Value"] = cotton_df["Value"].astype(str).str.replace(",", "").astype(float)

# Split "Data Item" into SubCategory + Measurement
cotton_df[["SubCategory", "Measurement"]] = cotton_df["Data Item"].str.split(" - ", expand=True)
cotton_df.drop(columns=["Data Item"], inplace=True)

# Normalize production units:
# NASS reports cotton production in 480-lb bales, even if sometimes written simply as "BALES"
cotton_df.loc[cotton_df["Measurement"].str.contains("PRODUCTION", case=False, na=False), "Unit"] = "480_lb_bales"

# -----------------------
# 2. Separate Area, Production, Yield
# -----------------------
area_df = cotton_df[cotton_df["Measurement"].str.contains("ACRES HARVESTED", case=False, na=False)].copy()
prod_df = cotton_df[cotton_df["Measurement"].str.contains("PRODUCTION", case=False, na=False)].copy()
yield_df = cotton_df[cotton_df["Measurement"].str.contains("YIELD", case=False, na=False)].copy()

# -----------------------
# 3. Aggregate by Year, County, SubCategory
# -----------------------
area_agg = area_df.groupby(["Year", "County", "SubCategory"], as_index=False).agg(Area=("Value", "sum"))
prod_agg = prod_df.groupby(["Year", "County", "SubCategory"], as_index=False).agg(Production=("Value", "sum"))
yield_agg = yield_df.groupby(["Year", "County", "SubCategory"], as_index=False).agg(Yield=("Value", "mean"))

# -----------------------
# 4. Merge all pieces
# -----------------------
merged = pd.merge(area_agg, prod_agg, on=["Year", "County", "SubCategory"], how="outer")
merged = pd.merge(merged, yield_agg, on=["Year", "County", "SubCategory"], how="outer")

# -----------------------
# 5. Yield reconciliation function
# -----------------------
def reconcile_yield(area, prod, reported_yield, unit="480_lb_bales", tolerance=0.05):
    """
    Decide final yield value using reported and computed yields.
    - Prefer reported yield if available.
    - If both reported and computed exist, warn if they differ > tolerance (default 5%).
    - If no reported yield, compute from production and area.
    """
    computed_yield = None
    if pd.notna(prod) and pd.notna(area) and area > 0:
        if unit == "480_lb_bales":
            computed_yield = (prod * 480) / area  # convert bales to lb, then per acre

    if pd.notna(reported_yield):
        if computed_yield is not None:
            diff = abs(computed_yield - reported_yield) / reported_yield
            if diff > tolerance:
                print(f"⚠️ Warning: Reported vs Computed yield differ by {diff:.1%}")
        return reported_yield
    else:
        return computed_yield

# -----------------------
# 6. County-level aggregation
# -----------------------
calc_result = []
for (year, county), group in merged.groupby(["Year", "County"]):
    # If PIMA/UPLAND exist, use them instead of total "COTTON"
    subcat_rows = group[group["SubCategory"] != "COTTON"]
    rows_to_use = subcat_rows if len(subcat_rows) > 0 else group

    area = rows_to_use["Area"].sum()
    prod = rows_to_use["Production"].sum()
    reported_yield = rows_to_use["Yield"].mean()

    final_yield = reconcile_yield(area, prod, reported_yield, unit="480_lb_bales")

    calc_result.append({
        "Season_year": year,
        "County": county,
        "Yield/acre": final_yield
    })

cotton_yield = pd.DataFrame(calc_result)

# -----------------------
# 7. Save to CSV
# -----------------------
cotton_yield.to_csv("Data/YIELD_COTTON.csv", index=False)

print("Saved YIELD_COTTON.csv")
print(cotton_yield.head())


In [ ]:
# ======================================================
# CORN Yield Processing with Unit Normalization (Refactored)
# ======================================================
# -----------------------
# 1. Helpers
# -----------------------
def _extract_unit(descriptor: str):
    """
    Extract reporting unit from descriptor string.
    Returns:
        - 'BU'   for bushels
        - 'TONS' for tons
        - 'LB'   for pounds
        - 'ACRES' for area
        - None   if not detected
    """
    if pd.isna(descriptor):
        return None

    desc = descriptor.upper()
    match = re.search(r"MEASURED IN ([A-Z]+)", desc)
    if match:
        return match.group(1)
    if "ACRES HARVESTED" in desc:
        return "ACRES"
    return None


def convert_production_value(value, unit, silage_dm_factor=0.35, use_silage_dm=True):
    """
    Convert a production value to pounds.
    """
    if pd.isna(value) or pd.isna(unit):
        return np.nan
    if unit == "BU":
        return value * 56.0
    elif unit == "TONS":
        asfed = value * 2000.0
        return asfed * (silage_dm_factor if use_silage_dm else 1.0)
    elif unit == "LB":
        return value
    return np.nan


def reconcile_yield(area, prod, reported_yield, unit,
                    silage_dm_factor=0.35, use_silage_dm=True, tolerance=0.05):
    """
    Prefer reported yield if available, but compare with computed yield.
    Uses converted production values.
    """
    prod_lb = convert_production_value(prod, unit, silage_dm_factor, use_silage_dm)
    computed_yield = prod_lb / area if (pd.notna(prod_lb) and pd.notna(area) and area > 0) else None

    if pd.notna(reported_yield):
        if computed_yield is not None:
            diff = abs(computed_yield - reported_yield) / reported_yield
            if diff > tolerance:
                print(f"⚠️ Warning: Reported vs Computed yield differ by {diff:.1%}")
        return reported_yield
    else:
        return computed_yield


# -----------------------
# 2. Load & clean raw corn data
# -----------------------
def load_corn_raw(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path, dtype=str)

    # Clean Value
    df["Value"] = df["Value"].str.replace(",", "", regex=True).astype(float)

    # Split "Data Item" into category parts
    parts = df["Data Item"].str.split(" - ", n=1, expand=True)
    df["Raw_SubCat"] = parts[0].str.strip()
    df["Descriptor"] = parts[1].str.strip()

    # Normalize subcategory name
    df["SubCategory"] = (
        df["Raw_SubCat"]
        .str.replace(r"^CORN,\s*", "", regex=True)
        .str.strip()
    )

    # Flags
    df["Is_Area"] = df["Descriptor"].str.contains("ACRES HARVESTED", case=False, na=False)
    df["Is_Prod"] = df["Descriptor"].str.contains("PRODUCTION", case=False, na=False)
    df["Is_Yield"] = df["Descriptor"].str.contains("YIELD", case=False, na=False)

    # Unit extraction
    df["Unit"] = df["Descriptor"].apply(_extract_unit)

    return df


# -----------------------
# 3. Yield computation
# -----------------------
def compute_corn_yield(csv_path: Path, silage_dm_factor=0.35, use_silage_dm=True):
    """
    Compute corn yield at county-year level with reconciliation of reported vs computed yields.
    """
    raw = load_corn_raw(csv_path)

    # Area
    area = (
        raw[raw["Is_Area"]]
        .groupby(["Year", "County", "SubCategory"], as_index=False)
        .agg(Area=("Value", "sum"))
    )

    # Production
    prod = (
        raw[raw["Is_Prod"]]
        .groupby(["Year", "County", "SubCategory", "Unit"], as_index=False)
        .agg(Production=("Value", "sum"))
    )

    # Reported yield
    yield_agg = (
        raw[raw["Is_Yield"]]
        .groupby(["Year", "County", "SubCategory"], as_index=False)
        .agg(Reported_Yield=("Value", "mean"))
    )

    # Merge all pieces
    merged = pd.merge(area, prod, on=["Year", "County", "SubCategory"], how="outer")
    merged = pd.merge(merged, yield_agg, on=["Year", "County", "SubCategory"], how="outer")

    # Add converted production (lb)
    merged["Production_lb"] = merged.apply(
        lambda row: convert_production_value(row["Production"], row["Unit"],
                                             silage_dm_factor, use_silage_dm),
        axis=1
    )

    # Reconcile yield per subcategory
    merged["Final_Yield"] = merged.apply(
        lambda row: reconcile_yield(
            row["Area"], row["Production"], row["Reported_Yield"], row["Unit"],
            silage_dm_factor=silage_dm_factor, use_silage_dm=use_silage_dm
        ),
        axis=1
    )

    # -----------------------
    # County-level aggregation
    # -----------------------
    results = []
    for (year, county), group in merged.groupby(["Year", "County"]):
        total_area = group["Area"].sum()
        total_prod_lb = group["Production_lb"].sum()

        # Weighted county yield
        computed_yield = total_prod_lb / total_area if total_area > 0 else np.nan

        # Reconcile with reported yield
        reported = group["Reported_Yield"].mean()
        if pd.notna(reported) and pd.notna(computed_yield):
            diff = abs(computed_yield - reported) / reported
            if diff > 0.05:
                print(f"⚠️ Warning: Reported vs Computed county yield differ by {diff:.1%}")
            final_yield = reported
        else:
            final_yield = reported if pd.notna(reported) else computed_yield

        results.append({
            "Season_year": year,
            "County": county,
            "Yield/acre": final_yield
        })

    county_yield = pd.DataFrame(results)
    return merged, county_yield


# -----------------------
# 4. Run for current dataset
# -----------------------
corn_path = Path("Data/Raw_Corn.csv")
merged_detail, county_yield = compute_corn_yield(
    corn_path,
    silage_dm_factor=0.35,
    use_silage_dm=True
)

print("Per-subcategory detail:")
display(merged_detail)

print("\nCounty overall (lb/acre):")
display(county_yield)

# Save to CSV
county_yield.to_csv("Data/YIELD_CORN.csv", index=False)
print("Saved YIELD_CORN.csv")


In [ ]:
# ======================================================
# BARLEY Yield Processing
# ======================================================
# -----------------------
# 1. Load raw barley data
# -----------------------
barley_df = pd.read_csv("Data/Raw_Barley.csv")

# Clean Value column
barley_df["Value"] = barley_df["Value"].str.replace(",", "", regex=True).astype(float)

# Split Data Item into SubCategory + Measurement
barley_df[["SubCategory", "Measurement"]] = barley_df["Data Item"].str.split(" - ", expand=True)
barley_df.drop(columns=["Data Item"], inplace=True)

# -----------------------
# 2. Separate Area, Production, Yield
# -----------------------
area_df = barley_df[barley_df["Measurement"].str.contains("ACRES HARVESTED", case=False, na=False)].copy()
prod_df = barley_df[barley_df["Measurement"].str.contains("PRODUCTION", case=False, na=False)].copy()
yield_df = barley_df[barley_df["Measurement"].str.contains("YIELD", case=False, na=False)].copy()

# Aggregate
area_agg = area_df.groupby(["Year", "County", "Commodity", "SubCategory"], as_index=False).agg(Area=("Value", "sum"))
prod_agg = prod_df.groupby(["Year", "County", "Commodity", "SubCategory"], as_index=False).agg(Production=("Value", "sum"))
yield_agg = yield_df.groupby(["Year", "County", "Commodity", "SubCategory"], as_index=False).agg(Reported_Yield=("Value", "mean"))

# Merge all pieces
merged = pd.merge(area_agg, prod_agg, on=["Year", "County", "Commodity", "SubCategory"], how="outer")
merged = pd.merge(merged, yield_agg, on=["Year", "County", "Commodity", "SubCategory"], how="outer")

# -----------------------
# 3. Calculate Yield
# -----------------------
calc_result = []
for (year, county, commodity), group in merged.groupby(["Year", "County", "Commodity"]):
    # Prefer subcategories (SPRING/WINTER) over total BARLEY
    subcat_rows = group[group["SubCategory"] != "BARLEY"]
    chosen = subcat_rows if len(subcat_rows) > 0 else group

    area = chosen["Area"].sum()
    prod = chosen["Production"].sum()
    reported = chosen["Reported_Yield"].mean()

    # Computed yield
    computed_yield = prod / area if area > 0 else np.nan

    # Reconcile
    if pd.notna(reported) and pd.notna(computed_yield):
        diff = abs(computed_yield - reported) / reported
        if diff > 0.05:
            print(f"⚠️ Warning: Reported vs Computed yield differ by {diff:.1%} "
                  f"(Year={year}, County={county}, Commodity={commodity})")
        final_yield = reported
    else:
        final_yield = reported if pd.notna(reported) else computed_yield

    calc_result.append({
        "Season_year": year,
        "County": county,
        "Yield/acre": final_yield
    })

calc_yield_df = pd.DataFrame(calc_result)

# -----------------------
# 4. Save final output
# -----------------------
out_path = Path("Data/YIELD_BARLEY.csv")
calc_yield_df.to_csv(out_path, index=False)
print(f"Saved {out_path}")
print(calc_yield_df)


In [ ]:
# ======================================================
# WHEAT Yield Processing
# ======================================================
# -----------------------
# 1. Load raw wheat data
# -----------------------
wheat_df = pd.read_csv("Data/Raw_Wheat.csv")

# Clean Value column
wheat_df["Value"] = wheat_df["Value"].str.replace(",", "", regex=True).astype(float)

# Split Data Item into SubCategory + Measurement
wheat_df[["SubCategory", "Measurement"]] = wheat_df["Data Item"].str.split(" - ", expand=True)
wheat_df.drop(columns=["Data Item"], inplace=True)

# -----------------------
# 2. Separate Area, Production, Yield
# -----------------------
area_df = wheat_df[wheat_df["Measurement"].str.contains("ACRES HARVESTED", case=False, na=False)].copy()
prod_df = wheat_df[wheat_df["Measurement"].str.contains("PRODUCTION", case=False, na=False)].copy()
yield_df = wheat_df[wheat_df["Measurement"].str.contains("YIELD", case=False, na=False)].copy()

# Aggregate
area_agg = area_df.groupby(["Year", "County", "Commodity", "SubCategory"], as_index=False).agg(Area=("Value", "sum"))
prod_agg = prod_df.groupby(["Year", "County", "Commodity", "SubCategory"], as_index=False).agg(Production=("Value", "sum"))
yield_agg = yield_df.groupby(["Year", "County", "Commodity", "SubCategory"], as_index=False).agg(Reported_Yield=("Value", "mean"))

# Merge all pieces
merged = pd.merge(area_agg, prod_agg, on=["Year", "County", "Commodity", "SubCategory"], how="outer")
merged = pd.merge(merged, yield_agg, on=["Year", "County", "Commodity", "SubCategory"], how="outer")

# -----------------------
# 3. Calculate Yield with reconciliation
# -----------------------
calc_result = []
for (year, county, commodity), group in merged.groupby(["Year", "County", "Commodity"]):
    # Prefer subcategories (e.g., SPRING, WINTER) over total WHEAT
    subcat_rows = group[group["SubCategory"] != "WHEAT"]
    chosen = subcat_rows if len(subcat_rows) > 0 else group

    area = chosen["Area"].sum()
    prod = chosen["Production"].sum()
    reported = chosen["Reported_Yield"].mean()

    # Computed yield
    computed_yield = prod / area if area > 0 else np.nan

    # Reconcile
    if pd.notna(reported) and pd.notna(computed_yield):
        diff = abs(computed_yield - reported) / reported
        if diff > 0.05:
            print(f"⚠️ Warning: Reported vs Computed yield differ by {diff:.1%} "
                  f"(Year={year}, County={county}, Commodity={commodity})")
        final_yield = reported
    else:
        final_yield = reported if pd.notna(reported) else computed_yield

    calc_result.append({
        "Season_year": year,
        "County": county,
        "Commodity": commodity,
        "Yield/acre": final_yield
    })

calc_yield_df = pd.DataFrame(calc_result)

# -----------------------
# 4. Save final output
# -----------------------
out_path = Path("Data/YIELD_WHEAT.csv")
calc_yield_df.to_csv(out_path, index=False)

print(f"Saved {out_path}")
print(calc_yield_df.head())


In [ ]:
# ======================================================
# ALFALFA Yield Processing
# ======================================================
# -----------------------
# 1. Load raw alfalfa data
# -----------------------
alfalfa_df = pd.read_csv("Data/Raw_Alfalfa.csv")

# Clean Value column
alfalfa_df["Value"] = alfalfa_df["Value"].str.replace(",", "", regex=True).astype(float)

# Split Data Item into SubCategory + Measurement
alfalfa_df[["SubCategory", "Measurement"]] = alfalfa_df["Data Item"].str.split(" - ", expand=True)
alfalfa_df.drop(columns=["Data Item"], inplace=True)

# -----------------------
# 2. Separate Area, Production, Yield
# -----------------------
area_df = alfalfa_df[alfalfa_df["Measurement"].str.contains("ACRES HARVESTED", case=False, na=False)].copy()
prod_df = alfalfa_df[alfalfa_df["Measurement"].str.contains("PRODUCTION", case=False, na=False)].copy()
yield_df = alfalfa_df[alfalfa_df["Measurement"].str.contains("YIELD", case=False, na=False)].copy()

# Aggregate
area_agg = area_df.groupby(["Year", "County", "Commodity", "SubCategory"], as_index=False).agg(Area=("Value", "sum"))
prod_agg = prod_df.groupby(["Year", "County", "Commodity", "SubCategory"], as_index=False).agg(Production=("Value", "sum"))
yield_agg = yield_df.groupby(["Year", "County", "Commodity", "SubCategory"], as_index=False).agg(Reported_Yield=("Value", "mean"))

# Merge
merged = pd.merge(area_agg, prod_agg, on=["Year", "County", "Commodity", "SubCategory"], how="outer")
merged = pd.merge(merged, yield_agg, on=["Year", "County", "Commodity", "SubCategory"], how="outer")

# -----------------------
# 3. Calculate Yield with reconciliation
# -----------------------
result_list = []
for (year, county), group in merged.groupby(["Year", "County"]):
    area = group["Area"].sum()
    prod = group["Production"].sum()
    reported = group["Reported_Yield"].mean()

    # Computed yield
    computed_yield = prod / area if area > 0 else np.nan

    # Reconcile
    if pd.notna(reported) and pd.notna(computed_yield):
        diff = abs(computed_yield - reported) / reported
        if diff > 0.05:
            print(f"⚠️ Warning: Reported vs Computed yield differ by {diff:.1%} "
                  f"(Year={year}, County={county}, Commodity={commodity})")
        final_yield = reported
    else:
        final_yield = reported if pd.notna(reported) else computed_yield

    result_list.append({
        "Season_year": year,
        "County": county,
        "Yield/acre": final_yield
    })

    #drop rows with NaN or 0.000 yield
    result_list = [res for res in result_list if pd.notna(res["Yield/acre"]) and res["Yield/acre"] != 0.0]

alfalfa_yield_df = pd.DataFrame(result_list)

# -----------------------
# 4. Save final output
# -----------------------
out_path = Path("Data/YIELD_ALFALFA.csv")
alfalfa_yield_df.to_csv(out_path, index=False)

print(f"Saved {out_path}")
print(alfalfa_yield_df)
